# Unit 3 group project: TrashNet + DINOv3 ConvNeXt Tiny

A minimal starting point: load the dataset, display one labeled image, load the pretrained CNN, and verify one forward pass. Training and evaluation are left for the group.

- Dataset: [garythung/trashnet on Hugging Face](https://huggingface.co/datasets/garythung/trashnet).
- Model: [DINOv3 ConvNeXt Tiny on Kaggle](https://www.kaggle.com/models/mataeoanderson/dinov3-convnext/keras/tiny), **version 1**.

On Kaggle, enable **Internet** and attach only `mataeoanderson/dinov3-convnext/keras/tiny/1`. The local, Git-ignored `kernel-metadata.json` configures both for publishing from this checkout. CPU is sufficient for this smoke test. Run all cells from top to bottom.

## 1. Dependencies and configuration

Install before importing TensorFlow. The model was saved with TensorFlow 2.21 / Keras 3.15; these versions match its serialization format. Downloaded data and weights stay outside Git.

In [ ]:
# Match the saved model format and pin data-loading dependencies for reproducibility.
%pip install -q tensorflow==2.21.0 keras==3.15.1 datasets==5.1.0 kagglehub==1.0.2 Pillow==12.3.0

In [ ]:
import os
from pathlib import Path

# Set the backend before importing Keras, and reduce TensorFlow startup logging.
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import json
import numpy as np
import tensorflow as tf
import keras
import kagglehub
from datasets import ClassLabel, Image, load_dataset
from IPython.display import display

DATASET_ID = "garythung/trashnet"
DATASET_REVISION = "1d4566ad6b22d2b5ef4eeba8f351b825d163eb11"
MODEL_HANDLE = "mataeoanderson/dinov3-convnext/keras/tiny/1"
MODEL_FILENAME = "dinov3_convnext_tiny.keras"
IMAGE_SIZE = (224, 224)
SAMPLE_INDEX = 0

# Relative paths work in Kaggle's writable working directory and local Jupyter.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

## 2. Load TrashNet

Load the full training split from Hugging Face's published Parquet conversion of the same dataset. All four shard filenames and the conversion commit were verified using the Hub's repository metadata. Using Parquet preserves the published label schema and avoids the image-folder loader's label-inference error on this repository.

`load_dataset("parquet", ..., split="train")` returns a `Dataset`. Its `image` feature decodes to a PIL image, and `label` is an integer described by `ClassLabel`. The class names come from that schema. No train/validation/test split is created yet.

API reference: [Hugging Face Parquet loading](https://huggingface.co/docs/datasets/en/loading#parquet).

In [ ]:
# Explicit published shards avoid guessing labels from the source folder layout.
# This commit pins Hugging Face's Parquet conversion, including its image/label schema.
parquet_base = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}"
parquet_files = [
    f"{parquet_base}/default/train/0000.parquet",
    f"{parquet_base}/default/train/0001.parquet",
    f"{parquet_base}/default/train/0002.parquet",
    f"{parquet_base}/default/train/0003.parquet",
]
dataset = load_dataset("parquet", data_files={"train": parquet_files}, split="train")
assert isinstance(dataset.features["image"], Image)
label_feature = dataset.features["label"]
assert isinstance(label_feature, ClassLabel)
class_names = label_feature.names

print(dataset)
print("Classes:", class_names)
print("Number of images:", len(dataset))

## 3. Show one image and its label

Read the label through `ClassLabel.int2str`; the display is the original RGB image, before model normalization.

In [ ]:
# A row contains image (PIL) and label (integer); any other columns are unused.
sample = dataset[SAMPLE_INDEX]
image = sample["image"].convert("RGB")
label_id = int(sample["label"])
label_name = label_feature.int2str(label_id)

print(f"Sample {SAMPLE_INDEX}: {label_name} (label {label_id})")
print("Image size:", image.size)
display(image)
# Save the same image so a completed Kaggle run has a downloadable preview.
image.save(OUTPUT_DIR / "sample_image.png")

## 4. Load only the Tiny CNN

`kagglehub.model_download` returns the local directory for the exact version. On Kaggle it resolves the attached model input; locally it downloads into KaggleHub's cache. The `.keras` file contains the architecture and weights and requires no custom layers.

The backbone produces features, not TrashNet class predictions. Freeze it as a starting point for the group's future classification head.

API references: [KaggleHub model download](https://github.com/Kaggle/kagglehub#download-models), [Keras model loading](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/).

In [ ]:
# Use the documented handle and filename; do not search for or load other variants.
model_dir = Path(kagglehub.model_download(MODEL_HANDLE))
model_file = model_dir / MODEL_FILENAME
if not model_file.is_file():
    raise FileNotFoundError(f"Expected model artifact: {MODEL_FILENAME}")

backbone = tf.keras.models.load_model(model_file, compile=False)
backbone.trainable = False
print("Loaded:", MODEL_HANDLE)
print("Model file:", model_file.name)
print("Parameters:", f"{backbone.count_params():,}")

## 5. Verify one forward pass

Follow the model card: resize to 224 × 224, scale RGB values to 0–1, then normalize with ImageNet mean and standard deviation. Input shape is `(batch, height, width, channels)`.

The documented output dictionary contains `pooled` (one 768-dimensional vector per image) and `feature_map` (7 × 7 × 768 spatial features for this input). Both are checked here. No classifier or training loop is added.

In [ ]:
def preprocess_image(rgb_image):
    """Prepare one PIL RGB image using the Tiny model card's normalization."""
    pixels = tf.convert_to_tensor(np.asarray(rgb_image), dtype=tf.float32)
    pixels = tf.image.resize(pixels, IMAGE_SIZE) / 255.0
    mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
    std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
    return ((pixels - mean) / std)[None, ...]

inputs = preprocess_image(image)
features = backbone(inputs, training=False)
# Handle both documented outputs explicitly; additional model fields are unused.
pooled = features["pooled"]
feature_map = features["feature_map"]
assert tuple(pooled.shape) == (1, 768)
assert tuple(feature_map.shape) == (1, 7, 7, 768)
tf.debugging.assert_all_finite(pooled, "Non-finite pooled features")
tf.debugging.assert_all_finite(feature_map, "Non-finite spatial features")

print("Input shape:", tuple(inputs.shape))
print("Pooled feature shape:", tuple(pooled.shape))
print("Spatial feature shape:", tuple(feature_map.shape))
print("SUCCESS: dataset, labeled image, Tiny CNN, and forward pass verified.")

# A compact artifact records actual runtime evidence without copying data or weights.
report = {
    "dataset": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "num_images": len(dataset),
    "classes": class_names,
    "sample_index": SAMPLE_INDEX,
    "sample_label_id": label_id,
    "sample_label": label_name,
    "model": MODEL_HANDLE,
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "pooled_shape": list(pooled.shape),
    "feature_map_shape": list(feature_map.shape),
    "status": "passed",
}
(OUTPUT_DIR / "smoke_test.json").write_text(json.dumps(report, indent=2) + "\n")

## 6. Group project work — placeholders

- Decide on leakage-safe training, validation, and test splits after inspecting the dataset.
- Add augmentations and a six-class classification head.
- Choose a training strategy and evaluate predictions.

The loaded `dataset`, `class_names`, `backbone`, and `preprocess_image` are ready for those steps.